# 🐾 EDA — Adopción de Mascotas en Malasia
### Dataset: PetFinder Malaysia · Variable objetivo: `Fee` (MYR)

---
## 📑 Contenido

| # | Sección |
|---|---|
| 1 | ⚙️ Instalación |
| 2 | 📂 Carga de datos |
| 3 | 📋 Estadísticas descriptivas |
| 4 | 🔍 Calidad de datos |
| 5 | 📊 Frecuencia absoluta y relativa |
| 6 | 📦 Distribución general |
| 7 | 💰 Análisis de Fee |
| 8 | 🗺️ Análisis geográfico |
| 9 | 🔗 Correlación simple |
| 10 | 🔬 Correlaciones parciales |
| 11 | 💾 Exportar HTML |
| 12 | 📌 Conclusiones |


## ⚙️ 1 · Instalación y Configuración

Ejecutar la siguiente celda **solo una vez** si falta alguna librería.

In [1]:
# Ejecutar UNA SOLA VEZ si es necesario
# import sys
# !{sys.executable} -m pip install pandas plotly


In [2]:
import os, warnings
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_columns", 30)

print("✅ Librerías cargadas correctamente")


✅ Librerías cargadas correctamente


In [3]:
# ── PATHS ──────────────────────────────────────────────────────
CSV_PATH   = r"C:\Users\juanc\OneDrive\Desktop\test.csv"
OUTPUT_DIR = r"C:\Users\juanc\OneDrive\Desktop\EDA_Output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── PALETA ─────────────────────────────────────────────────────
PALETTE   = px.colors.qualitative.Bold
COLOR_DOG = "#4E79A7"
COLOR_CAT = "#F28E2B"
BG_COLOR  = "#F8F9FA"

print(f"✅ Output dir: {OUTPUT_DIR}")


✅ Output dir: C:\Users\juanc\OneDrive\Desktop\EDA_Output


In [4]:
# ── DICCIONARIOS DE DECODIFICACIÓN ─────────────────────────────
STATE_MAP = {
    41324:"Kedah",       41325:"Kelantan",    41326:"Kuala Lumpur",
    41327:"Labuan",      41330:"Melaka",      41332:"Negeri Sembilan",
    41335:"Pahang",      41336:"Perak",       41342:"Perlis",
    41345:"Pulau Pinang",41361:"Sabah",       41367:"Sarawak",
    41380:"Selangor",    41401:"Terengganu",  41415:"Johor",
}
STATE_COORDS = {
    "Kuala Lumpur":(3.1390,101.6869), "Terengganu":(5.3117,103.1324),
    "Labuan":(5.2831,115.2308),       "Melaka":(2.1896,102.2501),
    "Perak":(4.5921,101.0901),        "Negeri Sembilan":(2.7258,102.2378),
    "Kedah":(6.1184,100.3685),        "Kelantan":(6.1254,102.2381),
    "Pahang":(3.8126,103.3256),       "Sarawak":(1.5533,110.3592),
    "Pulau Pinang":(5.4141,100.3288), "Perlis":(6.4449,100.2048),
    "Johor":(1.9344,103.3587),        "Sabah":(5.9788,116.0753),
    "Selangor":(3.0738,101.5183),
}
TYPE_MAP       = {1:"Perro 🐕", 2:"Gato 🐈"}
GENDER_MAP     = {1:"Macho", 2:"Hembra", 3:"Mixto"}
VACCINATED_MAP = {1:"Sí", 2:"No", 3:"No seguro"}
HEALTH_MAP     = {1:"Saludable", 2:"Lesión leve", 3:"Lesión grave"}
MATURITY_MAP   = {1:"Pequeño", 2:"Mediano", 3:"Grande", 4:"Extra grande", 0:"N/E"}
FUR_MAP        = {1:"Corto", 2:"Mediano", 3:"Largo", 0:"N/E"}
COLOR_MAP      = {1:"Negro", 2:"Marrón", 3:"Dorado",
                  4:"Amarillo", 5:"Crema", 6:"Gris", 7:"Blanco"}
BREED_MAP      = {
    307:"Mixed Breed (Perro)", 266:"Dom. Short Hair (Gato)",
    265:"Dom. Medium Hair (Gato)", 285:"Persa (Gato)",
    264:"Dom. Long Hair (Gato)", 205:"Labrador", 299:"Shih Tzu",
    141:"Golden Retriever", 179:"Husky Siberiano", 292:"Schnauzer",
    109:"Bulldog", 243:"Poodle", 254:"Rottweiler", 218:"Maltés", 103:"Beagle",
}
print("✅ Diccionarios configurados")


✅ Diccionarios configurados


## 📂 2 · Carga y Limpieza de Datos

In [5]:
df = pd.read_csv(CSV_PATH)
print(f"Shape original: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head(3)


Shape original: 3,972 filas × 23 columnas


,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,FurLength,Vaccinated,Dewormed,Sterilized,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PetID,PhotoAmt
0,2,Dopey & Grey,8,266,266,1,2,6,7,1,1,1,1,2,1,2,0,41326,2ece3b2573dcdcebd774e635dca15fd9,0,"Dopey Age: 8mths old Male One half of a pair, ...",e2dfc2935,2.00
1,2,Chi Chi,36,285,264,2,1,4,7,2,3,1,1,1,2,1,0,41326,2ece3b2573dcdcebd774e635dca15fd9,0,"Please note that Chichi has been neutered, the...",f153b465f,1.00
2,2,Sticky,2,265,0,1,6,7,0,2,2,1,1,2,1,1,200,41326,e59c106e9912fa30c898976278c2e834,0,"Sticky, named such because of his tendency to ...",3c90f3f54,4.00


In [6]:
# Decodificar todas las variables categóricas
df["Type_Label"]       = df["Type"].map(TYPE_MAP)
df["Gender_Label"]     = df["Gender"].map(GENDER_MAP)
df["Vaccinated_Label"] = df["Vaccinated"].map(VACCINATED_MAP)
df["Dewormed_Label"]   = df["Dewormed"].map(VACCINATED_MAP)
df["Sterilized_Label"] = df["Sterilized"].map(VACCINATED_MAP)
df["Health_Label"]     = df["Health"].map(HEALTH_MAP)
df["Maturity_Label"]   = df["MaturitySize"].map(MATURITY_MAP)
df["Fur_Label"]        = df["FurLength"].map(FUR_MAP)
df["Color_Label"]      = df["Color1"].map(COLOR_MAP)
df["Breed_Label"]      = df["Breed1"].map(BREED_MAP).fillna("Otra raza")
df["State_Name"]       = df["State"].map(STATE_MAP)
df["Fee_Category"]     = pd.cut(df["Fee"],
                                bins=[-1, 0, 50, 200, 500, 1500],
                                labels=["Gratis","Bajo (1-50)","Medio (51-200)",
                                        "Alto (201-500)","Premium (>500)"])
print(f"✅ Shape final: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.dtypes.to_frame("dtype")


✅ Shape final: 3,972 filas × 35 columnas


,dtype
Type,int64
Name,object
Age,int64
Breed1,int64
Breed2,int64
Gender,int64
Color1,int64
Color2,int64
Color3,int64
MaturitySize,int64


## 📋 3 · Estadísticas Descriptivas

Resumen cuantitativo de las variables numéricas más relevantes.

In [7]:
numeric_cols = ["Age","Fee","Quantity","PhotoAmt","VideoAmt"]
df[numeric_cols].describe().round(2)


,Age,Fee,Quantity,PhotoAmt,VideoAmt
count,3972.00,3972.00,3972.00,3972.00,3972.00
mean,11.29,21.77,1.60,3.64,0.04
std,17.50,79.28,1.50,3.37,0.30
min,0.00,0.00,1.00,0.00,0.00
25%,2.00,0.00,1.00,1.00,0.00
50%,4.00,0.00,1.00,3.00,0.00
75%,12.00,0.00,1.00,5.00,0.00
max,156.00,1500.00,20.00,30.00,8.00


In [8]:
# KPIs clave del dataset
total      = len(df)
n_dogs     = (df["Type"]==1).sum()
n_cats     = (df["Type"]==2).sum()
pct_free   = (df["Fee"]==0).mean()*100
avg_fee    = df.loc[df["Fee"]>0,"Fee"].mean()
avg_age    = df["Age"].mean()
pct_health = (df["Health"]==1).mean()*100

kpis = {
    "Total mascotas"           : f"{total:,}",
    "Perros"                   : f"{n_dogs:,} ({n_dogs/total*100:.1f}%)",
    "Gatos"                    : f"{n_cats:,} ({n_cats/total*100:.1f}%)",
    "Adopciones gratuitas"     : f"{pct_free:.1f}%",
    "Fee promedio (con cargo)" : f"{avg_fee:.2f} MYR",
    "Edad promedio"            : f"{avg_age:.1f} meses",
    "Mascotas saludables"      : f"{pct_health:.1f}%",
    "Estados representados"    : df["State_Name"].nunique(),
}
pd.DataFrame(kpis.items(), columns=["Indicador","Valor"])


,Indicador,Valor
0,Total mascotas,"3,972"
1,Perros,"2,100 (52.9%)"
2,Gatos,"1,872 (47.1%)"
3,Adopciones gratuitas,85.5%
4,Fee promedio (con cargo),150.38 MYR
5,Edad promedio,11.3 meses
6,Mascotas saludables,96.2%
7,Estados representados,15


## 🔍 4 · Calidad de Datos y Valores Nulos

Solo `Name` y `Description` tienen nulos. Calidad general del dataset: **>99%**.

In [9]:
null_df = (df.isnull().sum()
             .reset_index()
             .rename(columns={"index":"Columna", 0:"Nulos"}))
null_df.columns = ["Columna","Nulos"]
null_df["% Nulos"] = (null_df["Nulos"]/len(df)*100).round(2)
null_df = null_df[null_df["Nulos"]>0].sort_values("Nulos",ascending=True)
print("Columnas con nulos:")
display(null_df)


Columnas con nulos:


,Columna,Nulos,% Nulos
20,Description,1,0.03
1,Name,414,10.42


In [10]:
fig = px.bar(null_df, x="% Nulos", y="Columna", orientation="h",
             title="Porcentaje de Valores Nulos por Columna",
             text="Nulos", color="% Nulos", color_continuous_scale="Reds",
             labels={"% Nulos":"% Nulos","Columna":""})
fig.update_traces(texttemplate="%{text} registros", textposition="outside")
fig.update_layout(height=280, showlegend=False, coloraxis_showscale=False,
                  plot_bgcolor=BG_COLOR, paper_bgcolor="white")
fig.show()


## 📊 5 · Análisis de Frecuencia Absoluta y Relativa

> **Frecuencia absoluta (FA):** cantidad de veces que aparece cada valor.  
> **Frecuencia relativa (FR):** proporción sobre el total (FA / N).  
> **Frecuencia acumulada (FAC / FRC):** suma progresiva de FA o FR.

Se analiza cada variable categórica clave del dataset.


In [11]:
def tabla_frecuencias(serie, nombre_col):
    """Genera tabla completa de frecuencias para una Serie categórica."""
    fa  = serie.value_counts().reset_index()
    fa.columns = [nombre_col, "Frec. Absoluta"]
    fa  = fa.sort_values("Frec. Absoluta", ascending=False).reset_index(drop=True)
    fa["Frec. Relativa (%)"] = (fa["Frec. Absoluta"] / fa["Frec. Absoluta"].sum() * 100).round(2)
    fa["Frec. Abs. Acum."]   = fa["Frec. Absoluta"].cumsum()
    fa["Frec. Rel. Acum. (%)"] = fa["Frec. Relativa (%)"].cumsum().round(2)
    fa.index = fa.index + 1   # índice desde 1
    fa.index.name = "Rango"
    return fa

print("✅ Función tabla_frecuencias() lista")


✅ Función tabla_frecuencias() lista


### 5.1 · Tipo de Mascota

In [12]:
tbl_tipo = tabla_frecuencias(df["Type_Label"], "Tipo")
display(tbl_tipo)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=tbl_tipo["Tipo"], y=tbl_tipo["Frec. Absoluta"],
    name="Frec. Absoluta",
    marker_color=[COLOR_DOG, COLOR_CAT],
    text=tbl_tipo["Frec. Absoluta"], textposition="outside",
    yaxis="y1"
))
fig.add_trace(go.Scatter(
    x=tbl_tipo["Tipo"], y=tbl_tipo["Frec. Rel. Acum. (%)"],
    name="Frec. Rel. Acum. (%)",
    mode="lines+markers+text",
    text=[f"{v:.1f}%" for v in tbl_tipo["Frec. Rel. Acum. (%)"]],
    textposition="top center",
    marker=dict(size=10, color="#E15759"),
    line=dict(color="#E15759", width=2),
    yaxis="y2"
))
fig.update_layout(
    title="Frecuencia Absoluta y Relativa Acumulada — Tipo de Mascota",
    yaxis=dict(title="Frec. Absoluta", showgrid=True),
    yaxis2=dict(title="Frec. Rel. Acum. (%)", overlaying="y",
                side="right", range=[0,115], showgrid=False),
    legend=dict(x=0.7, y=0.95), plot_bgcolor=BG_COLOR,
    paper_bgcolor="white", height=420
)
fig.show()


,Tipo,Frec. Absoluta,Frec. Relativa (%),Frec. Abs. Acum.,Frec. Rel. Acum. (%)
Rango,,,,,
1,Perro 🐕,2100,52.87,2100,52.87
2,Gato 🐈,1872,47.13,3972,100.00


### 5.2 · Género

In [13]:
tbl_genero = tabla_frecuencias(df["Gender_Label"], "Género")
display(tbl_genero)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=tbl_genero["Género"], y=tbl_genero["Frec. Absoluta"],
    name="Frec. Absoluta",
    marker_color=PALETTE[:3],
    text=tbl_genero["Frec. Absoluta"], textposition="outside",
    yaxis="y1"
))
fig.add_trace(go.Scatter(
    x=tbl_genero["Género"], y=tbl_genero["Frec. Rel. Acum. (%)"],
    name="Frec. Rel. Acum. (%)",
    mode="lines+markers+text",
    text=[f"{v:.1f}%" for v in tbl_genero["Frec. Rel. Acum. (%)"]],
    textposition="top center",
    marker=dict(size=10, color="#E15759"),
    line=dict(color="#E15759", width=2),
    yaxis="y2"
))
fig.update_layout(
    title="Frecuencia Absoluta y Relativa Acumulada — Género",
    yaxis=dict(title="Frec. Absoluta"),
    yaxis2=dict(title="Frec. Rel. Acum. (%)", overlaying="y",
                side="right", range=[0,115], showgrid=False),
    legend=dict(x=0.7, y=0.95), plot_bgcolor=BG_COLOR,
    paper_bgcolor="white", height=420
)
fig.show()


,Género,Frec. Absoluta,Frec. Relativa (%),Frec. Abs. Acum.,Frec. Rel. Acum. (%)
Rango,,,,,
1,Hembra,1878,47.28,1878,47.28
2,Macho,1504,37.87,3382,85.15
3,Mixto,590,14.85,3972,100.00


### 5.3 · Estado de Salud

In [14]:
tbl_salud = tabla_frecuencias(df["Health_Label"], "Salud")
display(tbl_salud)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=tbl_salud["Salud"], y=tbl_salud["Frec. Absoluta"],
    name="Frec. Absoluta",
    marker_color=["#59A14F","#EDC948","#E15759"],
    text=tbl_salud["Frec. Absoluta"], textposition="outside", yaxis="y1"
))
fig.add_trace(go.Scatter(
    x=tbl_salud["Salud"], y=tbl_salud["Frec. Rel. Acum. (%)"],
    name="Frec. Rel. Acum. (%)", mode="lines+markers+text",
    text=[f"{v:.1f}%" for v in tbl_salud["Frec. Rel. Acum. (%)"]],
    textposition="top center",
    marker=dict(size=10, color="#4E79A7"),
    line=dict(color="#4E79A7", width=2), yaxis="y2"
))
fig.update_layout(
    title="Frecuencia Absoluta y Relativa Acumulada — Estado de Salud",
    yaxis=dict(title="Frec. Absoluta"),
    yaxis2=dict(title="Frec. Rel. Acum. (%)", overlaying="y",
                side="right", range=[0,115], showgrid=False),
    legend=dict(x=0.7, y=0.95), plot_bgcolor=BG_COLOR,
    paper_bgcolor="white", height=420
)
fig.show()


,Salud,Frec. Absoluta,Frec. Relativa (%),Frec. Abs. Acum.,Frec. Rel. Acum. (%)
Rango,,,,,
1,Saludable,3820,96.17,3820,96.17
2,Lesión leve,143,3.60,3963,99.77
3,Lesión grave,9,0.23,3972,100.00


### 5.4 · Vacunación, Desparasitación y Esterilización

In [15]:
tbl_vax    = tabla_frecuencias(df["Vaccinated_Label"],  "Vacunado")
tbl_deworm = tabla_frecuencias(df["Dewormed_Label"],    "Desparasitado")
tbl_steril = tabla_frecuencias(df["Sterilized_Label"],  "Esterilizado")

print("── Vacunado ──")
display(tbl_vax)
print("\n── Desparasitado ──")
display(tbl_deworm)
print("\n── Esterilizado ──")
display(tbl_steril)


── Vacunado ──


,Vacunado,Frec. Absoluta,Frec. Relativa (%),Frec. Abs. Acum.,Frec. Rel. Acum. (%)
Rango,,,,,
1,No,1795,45.19,1795,45.19
2,Sí,1690,42.55,3485,87.74
3,No seguro,487,12.26,3972,100.00



── Desparasitado ──


,Desparasitado,Frec. Absoluta,Frec. Relativa (%),Frec. Abs. Acum.,Frec. Rel. Acum. (%)
Rango,,,,,
1,Sí,2350,59.16,2350,59.16
2,No,1186,29.86,3536,89.02
3,No seguro,436,10.98,3972,100.00



── Esterilizado ──


,Esterilizado,Frec. Absoluta,Frec. Relativa (%),Frec. Abs. Acum.,Frec. Rel. Acum. (%)
Rango,,,,,
1,No,2507,63.12,2507,63.12
2,Sí,941,23.69,3448,86.81
3,No seguro,524,13.19,3972,100.00


In [16]:
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=("Vacunado","Desparasitado","Esterilizado"))
COLOR_VAX = {"Sí":"#59A14F","No":"#E15759","No seguro":"#EDC948"}

for col_i, (tbl, col_name) in enumerate(
        [(tbl_vax,"Vacunado"),(tbl_deworm,"Desparasitado"),(tbl_steril,"Esterilizado")],
        start=1):
    fig.add_trace(go.Bar(
        x=tbl[col_name], y=tbl["Frec. Absoluta"],
        marker_color=[COLOR_VAX.get(v,"#999") for v in tbl[col_name]],
        text=[f"{v}<br>{r:.1f}%" for v,r in
              zip(tbl["Frec. Absoluta"],tbl["Frec. Relativa (%)"])],
        textposition="outside", showlegend=False
    ), row=1, col=col_i)

fig.update_layout(title="Frecuencias Tratamientos Sanitarios",
                  height=440, plot_bgcolor=BG_COLOR, paper_bgcolor="white")
fig.show()


### 5.5 · Color Primario

In [17]:
tbl_color = tabla_frecuencias(df["Color_Label"], "Color")
display(tbl_color)

COLOR_HEX = {
    "Negro":"#2d2d2d","Marrón":"#8B4513","Dorado":"#C8960C",
    "Amarillo":"#D4A017","Crema":"#A0785A","Gris":"#6B6B6B","Blanco":"#9E9E9E",
}
fig = go.Figure()
fig.add_trace(go.Bar(
    x=tbl_color["Color"], y=tbl_color["Frec. Absoluta"],
    name="Frec. Absoluta",
    marker=dict(
        color=[COLOR_HEX.get(c,"#999") for c in tbl_color["Color"]],
        line=dict(color="#222", width=1.5)
    ),
    text=tbl_color["Frec. Absoluta"], textposition="outside", yaxis="y1"
))
fig.add_trace(go.Scatter(
    x=tbl_color["Color"], y=tbl_color["Frec. Rel. Acum. (%)"],
    name="Frec. Rel. Acum. (%)", mode="lines+markers+text",
    text=[f"{v:.1f}%" for v in tbl_color["Frec. Rel. Acum. (%)"]],
    textposition="top center",
    marker=dict(size=9, color="#E15759"),
    line=dict(color="#E15759", width=2), yaxis="y2"
))
fig.update_layout(
    title="Frecuencia por Color Primario (con Frec. Rel. Acumulada)",
    yaxis=dict(title="Frec. Absoluta"),
    yaxis2=dict(title="Frec. Rel. Acum. (%)", overlaying="y",
                side="right", range=[0,120], showgrid=False),
    legend=dict(x=0.7, y=0.95), plot_bgcolor=BG_COLOR,
    paper_bgcolor="white", height=440
)
fig.show()


,Color,Frec. Absoluta,Frec. Relativa (%),Frec. Abs. Acum.,Frec. Rel. Acum. (%)
Rango,,,,,
1,Negro,1989,50.08,1989,50.08
2,Marrón,950,23.92,2939,74.00
3,Crema,262,6.60,3201,80.60
4,Dorado,250,6.29,3451,86.89
5,Gris,199,5.01,3650,91.90
6,Blanco,184,4.63,3834,96.53
7,Amarillo,138,3.47,3972,100.00


### 5.6 · Top 15 Razas — Diagrama de Pareto

In [18]:
tbl_raza = tabla_frecuencias(df["Breed_Label"], "Raza").head(15)
display(tbl_raza)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=tbl_raza["Raza"], y=tbl_raza["Frec. Absoluta"],
    name="Frec. Absoluta",
    marker_color="#4E79A7",
    text=tbl_raza["Frec. Absoluta"], textposition="outside", yaxis="y1"
))
fig.add_trace(go.Scatter(
    x=tbl_raza["Raza"], y=tbl_raza["Frec. Rel. Acum. (%)"],
    name="Frec. Rel. Acum. (%)", mode="lines+markers",
    marker=dict(size=8, color="#E15759"),
    line=dict(color="#E15759", width=2), yaxis="y2"
))
fig.add_hline(y=80, line_dash="dash", line_color="gray",
              annotation_text="Regla 80%", annotation_position="right",
              yref="y2")
fig.update_layout(
    title="Diagrama de Pareto — Top 15 Razas",
    xaxis_tickangle=-35,
    yaxis=dict(title="Frec. Absoluta"),
    yaxis2=dict(title="Frec. Rel. Acum. (%)", overlaying="y",
                side="right", range=[0,110], showgrid=False),
    legend=dict(x=0.6, y=0.95), plot_bgcolor=BG_COLOR,
    paper_bgcolor="white", height=480
)
fig.show()


,Raza,Frec. Absoluta,Frec. Relativa (%),Frec. Abs. Acum.,Frec. Rel. Acum. (%)
Rango,,,,,
1,Mixed Breed (Perro),1499,37.74,1499,37.74
2,Dom. Short Hair (Gato),1026,25.83,2525,63.57
3,Otra raza,502,12.64,3027,76.21
4,Dom. Medium Hair (Gato),365,9.19,3392,85.40
5,Persa (Gato),78,1.96,3470,87.36
6,Dom. Long Hair (Gato),71,1.79,3541,89.15
7,Labrador,63,1.59,3604,90.74
8,Shih Tzu,63,1.59,3667,92.33
9,Golden Retriever,54,1.36,3721,93.69


### 5.7 · Categoría de Fee

In [19]:
tbl_fee_cat = tabla_frecuencias(df["Fee_Category"].astype(str), "Categoría Fee")
display(tbl_fee_cat)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=tbl_fee_cat["Categoría Fee"], y=tbl_fee_cat["Frec. Absoluta"],
    name="Frec. Absoluta",
    marker_color=PALETTE[:len(tbl_fee_cat)],
    text=[f"{a}<br>({r:.1f}%)" for a,r in
          zip(tbl_fee_cat["Frec. Absoluta"],tbl_fee_cat["Frec. Relativa (%)"])],
    textposition="outside", yaxis="y1"
))
fig.add_trace(go.Scatter(
    x=tbl_fee_cat["Categoría Fee"], y=tbl_fee_cat["Frec. Rel. Acum. (%)"],
    name="Frec. Rel. Acum. (%)", mode="lines+markers+text",
    text=[f"{v:.1f}%" for v in tbl_fee_cat["Frec. Rel. Acum. (%)"]],
    textposition="top center",
    marker=dict(size=9, color="#2d2d2d"),
    line=dict(color="#2d2d2d", width=2), yaxis="y2"
))
fig.update_layout(
    title="Frecuencia por Categoría de Fee",
    yaxis=dict(title="Frec. Absoluta"),
    yaxis2=dict(title="Frec. Rel. Acum. (%)", overlaying="y",
                side="right", range=[0,115], showgrid=False),
    legend=dict(x=0.6, y=0.95), plot_bgcolor=BG_COLOR,
    paper_bgcolor="white", height=440
)
fig.show()


,Categoría Fee,Frec. Absoluta,Frec. Relativa (%),Frec. Abs. Acum.,Frec. Rel. Acum. (%)
Rango,,,,,
1,Gratis,3397,85.52,3397,85.52
2,Medio (51-200),251,6.32,3648,91.84
3,Bajo (1-50),209,5.26,3857,97.10
4,Alto (201-500),100,2.52,3957,99.62
5,Premium (>500),15,0.38,3972,100.00


## 📦 6 · Distribución General de Variables

### 6.1 · Tipo de mascota (gráfico de dona)

In [20]:
# FIX: usar go.Pie directamente — más estable que px.pie en todos los entornos
type_counts = df["Type_Label"].value_counts().reset_index()
type_counts.columns = ["Tipo","Cantidad"]

fig = go.Figure(go.Pie(
    labels=type_counts["Tipo"],
    values=type_counts["Cantidad"],
    hole=0.45,
    marker=dict(colors=[COLOR_DOG, COLOR_CAT],
                line=dict(color="white", width=2)),
    textinfo="label+percent+value",
    textfont_size=13,
    pull=[0.03, 0.03],
))
fig.update_layout(
    title="Distribución por Tipo de Mascota",
    height=420, paper_bgcolor="white",
    annotations=[dict(text="🐾", x=0.5, y=0.5,
                      font_size=30, showarrow=False)]
)
fig.show()


### 6.2 · Género

In [21]:
gender_counts = df["Gender_Label"].value_counts().reset_index()
gender_counts.columns = ["Género","Cantidad"]

fig = go.Figure(go.Bar(
    x=gender_counts["Género"], y=gender_counts["Cantidad"],
    marker_color=PALETTE[:3],
    text=gender_counts["Cantidad"], textposition="outside",
))
fig.update_layout(
    title="Distribución por Género",
    height=400, showlegend=False,
    plot_bgcolor=BG_COLOR, paper_bgcolor="white",
    xaxis_title="Género", yaxis_title="Cantidad"
)
fig.show()


### 6.3 · Distribución de Edad (histograma)

In [22]:
# FIX: usar go.Histogram por tipo separado — evita problema de color en px
dogs = df[df["Type"]==1]["Age"]
cats = df[df["Type"]==2]["Age"]

fig = go.Figure()
fig.add_trace(go.Histogram(x=dogs, nbinsx=40, name="Perro 🐕",
                           marker_color=COLOR_DOG, opacity=0.75))
fig.add_trace(go.Histogram(x=cats, nbinsx=40, name="Gato 🐈",
                           marker_color=COLOR_CAT, opacity=0.75))
fig.update_layout(
    barmode="overlay",
    title="Distribución de Edad (en meses) por Tipo",
    xaxis_title="Edad (meses)", yaxis_title="Cantidad",
    height=420, plot_bgcolor=BG_COLOR, paper_bgcolor="white",
    legend=dict(x=0.8, y=0.95)
)
fig.show()


### 6.4 · Top 12 Razas más frecuentes

In [23]:
top_breeds = df["Breed_Label"].value_counts().head(12).reset_index()
top_breeds.columns = ["Raza","Cantidad"]

fig = go.Figure(go.Bar(
    x=top_breeds["Cantidad"], y=top_breeds["Raza"],
    orientation="h",
    marker=dict(color=top_breeds["Cantidad"],
                colorscale="Blues", showscale=False,
                line=dict(color="#ccc",width=0.5)),
    text=top_breeds["Cantidad"], textposition="outside",
))
fig.update_layout(
    title="Top 12 Razas Más Frecuentes",
    height=460, plot_bgcolor=BG_COLOR, paper_bgcolor="white",
    xaxis_title="Cantidad", yaxis=dict(autorange="reversed")
)
fig.show()


### 6.5 · Color Primario

In [24]:
COLOR_HEX = {
    "Negro":"#2d2d2d","Marrón":"#8B4513","Dorado":"#C8960C",
    "Amarillo":"#D4A017","Crema":"#A0785A","Gris":"#6B6B6B","Blanco":"#9E9E9E",
}
color_counts = df["Color_Label"].value_counts().reset_index()
color_counts.columns = ["Color","Cantidad"]

fig = go.Figure(go.Bar(
    x=color_counts["Color"], y=color_counts["Cantidad"],
    marker=dict(
        color=[COLOR_HEX.get(c,"#999") for c in color_counts["Color"]],
        line=dict(color="#222222", width=1.5)   # borde oscuro en todos
    ),
    text=color_counts["Cantidad"], textposition="outside",
))
fig.update_layout(
    title="Color Primario de las Mascotas",
    height=420, showlegend=False,
    plot_bgcolor=BG_COLOR, paper_bgcolor="white",
    xaxis_title="Color", yaxis_title="Cantidad"
)
fig.show()


### 6.6 · Salud, Tamaño y Pelaje

In [25]:
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=("Estado de Salud","Tamaño de Madurez",
                                    "Longitud de Pelaje"))
for col_i, col_label in enumerate(["Health_Label","Maturity_Label","Fur_Label"],1):
    vc = df[col_label].value_counts().reset_index()
    vc.columns = ["label","count"]
    fig.add_trace(go.Bar(x=vc["label"], y=vc["count"],
                         marker_color=PALETTE[:len(vc)],
                         text=vc["count"], textposition="outside",
                         showlegend=False), row=1, col=col_i)
fig.update_layout(height=420, paper_bgcolor="white",
                  plot_bgcolor=BG_COLOR, title_text="Salud, Tamaño y Pelaje")
fig.show()


### 6.7 · Estado Sanitario (Vacunación / Desparasitación / Esterilización)

In [26]:
vax_data = []
for lbl, col in [("Vacunado","Vaccinated_Label"),
                  ("Desparasitado","Dewormed_Label"),
                  ("Esterilizado","Sterilized_Label")]:
    for _, row in df[col].value_counts().reset_index().iterrows():
        vax_data.append({"Tratamiento":lbl,"Respuesta":row[col],"Cantidad":row["count"]})
vax_df = pd.DataFrame(vax_data)

fig = px.bar(vax_df, x="Tratamiento", y="Cantidad", color="Respuesta",
             barmode="group",
             title="Estado Sanitario — Vacunación, Desparasitación y Esterilización",
             color_discrete_map={"Sí":"#59A14F","No":"#E15759","No seguro":"#EDC948"},
             text="Cantidad")
fig.update_traces(textposition="outside")
fig.update_layout(height=440, plot_bgcolor=BG_COLOR, paper_bgcolor="white")
fig.show()


## 💰 7 · Análisis de la Variable Objetivo: Fee de Adopción

> **85.5%** de las adopciones son gratuitas (mediana = 0 MYR).  
> Cuando hay fee, el promedio sube a **~150 MYR** (~33 USD).

### 7.1 · Distribución del Fee


In [27]:
# FIX: go.Histogram directamente — evita problema con px.histogram + color
df_fee = df[df["Fee"] <= 600]
dogs_fee = df_fee[df_fee["Type"]==1]["Fee"]
cats_fee = df_fee[df_fee["Type"]==2]["Fee"]
mediana  = df["Fee"].median()

fig = go.Figure()
fig.add_trace(go.Histogram(x=dogs_fee, nbinsx=50, name="Perro 🐕",
                           marker_color=COLOR_DOG, opacity=0.75))
fig.add_trace(go.Histogram(x=cats_fee, nbinsx=50, name="Gato 🐈",
                           marker_color=COLOR_CAT, opacity=0.75))
fig.add_vline(x=mediana, line_dash="dash", line_color="red",
              annotation_text=f"Mediana: {mediana:.0f} MYR",
              annotation_position="top right")
fig.update_layout(
    barmode="overlay",
    title="Distribución del Fee de Adopción (Fee ≤ 600 MYR)",
    xaxis_title="Fee (MYR)", yaxis_title="Cantidad",
    height=440, plot_bgcolor=BG_COLOR, paper_bgcolor="white",
    legend=dict(x=0.75, y=0.95)
)
fig.show()


### 7.2 · Categorías del Fee

In [28]:
fee_cat = df["Fee_Category"].value_counts().reset_index()
fee_cat.columns = ["Categoría","Cantidad"]

fig = go.Figure(go.Pie(
    labels=fee_cat["Categoría"], values=fee_cat["Cantidad"],
    hole=0.38,
    marker=dict(colors=PALETTE[:len(fee_cat)],
                line=dict(color="white", width=2)),
    textinfo="label+percent+value",
    textfont_size=11,
    pull=[0.04]*len(fee_cat),
))
fig.update_layout(title="Categorías del Fee de Adopción",
                  height=440, paper_bgcolor="white")
fig.show()


### 7.3 · Fee por Tipo de Mascota y Estado de Salud

In [29]:
fig = go.Figure()
for tipo, color in [("Perro 🐕", COLOR_DOG), ("Gato 🐈", COLOR_CAT)]:
    datos = df[(df["Type_Label"]==tipo) & (df["Fee"]>0)]["Fee"]
    fig.add_trace(go.Box(y=datos, name=tipo, marker_color=color,
                         boxpoints="outliers", jitter=0.3))
fig.update_layout(
    title="Distribución del Fee por Tipo (solo registros con Fee > 0)",
    yaxis_title="Fee (MYR)", height=440,
    plot_bgcolor=BG_COLOR, paper_bgcolor="white", showlegend=True
)
fig.show()


In [30]:
fig = go.Figure()
for salud, color in [("Saludable","#59A14F"),
                     ("Lesión leve","#EDC948"),
                     ("Lesión grave","#E15759")]:
    datos = df[df["Health_Label"]==salud]["Fee"]
    fig.add_trace(go.Box(y=datos, name=salud, marker_color=color,
                         boxpoints="outliers"))
fig.update_layout(
    title="Fee según Estado de Salud",
    yaxis_title="Fee (MYR)", height=440,
    plot_bgcolor=BG_COLOR, paper_bgcolor="white"
)
fig.show()


### 7.4 · Fee promedio por Estado

In [31]:
fee_state = (df.groupby("State_Name")["Fee"]
               .agg(Promedio="mean", Mediana="median", Registros="count")
               .reset_index()
               .rename(columns={"State_Name":"Estado"}))
fee_state["Promedio"] = fee_state["Promedio"].round(2)

fig = go.Figure(go.Bar(
    x=fee_state.sort_values("Promedio",ascending=False)["Estado"],
    y=fee_state.sort_values("Promedio",ascending=False)["Promedio"],
    marker=dict(color=fee_state.sort_values("Promedio",ascending=False)["Promedio"],
                colorscale="Teal", showscale=True,
                colorbar=dict(title="Fee MYR")),
    text=fee_state.sort_values("Promedio",ascending=False)["Promedio"],
    texttemplate="%{text:.1f}", textposition="outside",
))
fig.update_layout(
    title="Fee Promedio de Adopción por Estado",
    height=460, plot_bgcolor=BG_COLOR, paper_bgcolor="white",
    xaxis_tickangle=-30, xaxis_title="Estado", yaxis_title="Fee Promedio (MYR)"
)
fig.show()


### 7.5 · Relación Fotos vs Fee

In [32]:
# FIX: sin trendline='ols' (requiere statsmodels no instalado)
# Calculamos la línea de tendencia manualmente con numpy
x_vals = df["PhotoAmt"].values
y_vals = df["Fee"].values
mask   = ~(np.isnan(x_vals) | np.isnan(y_vals))
coef   = np.polyfit(x_vals[mask], y_vals[mask], 1)
x_line = np.linspace(x_vals[mask].min(), x_vals[mask].max(), 100)
y_line = np.polyval(coef, x_line)

fig = go.Figure()
for tipo, color in [("Perro 🐕", COLOR_DOG), ("Gato 🐈", COLOR_CAT)]:
    sub = df[df["Type_Label"]==tipo]
    fig.add_trace(go.Scatter(
        x=sub["PhotoAmt"], y=sub["Fee"], mode="markers",
        name=tipo, marker=dict(color=color, opacity=0.4, size=5)
    ))
fig.add_trace(go.Scatter(
    x=x_line, y=y_line, mode="lines",
    name=f"Tendencia (slope={coef[0]:.2f})",
    line=dict(color="black", dash="dash", width=2)
))
fig.update_layout(
    title="Relación: Cantidad de Fotos vs Fee de Adopción",
    xaxis_title="Cantidad de Fotos", yaxis_title="Fee (MYR)",
    height=460, plot_bgcolor=BG_COLOR, paper_bgcolor="white"
)
fig.show()


## 🗺️ 8 · Análisis Geográfico — Estados de Malasia

### 8.1 · Resumen por estado

In [33]:
state_summary = (df.groupby("State_Name").agg(
    Total       =("PetID",  "count"),
    Perros      =("Type",   lambda x: (x==1).sum()),
    Gatos       =("Type",   lambda x: (x==2).sum()),
    Fee_Promedio=("Fee",    "mean"),
    Pct_Gratis  =("Fee",    lambda x: (x==0).mean()*100),
    Edad_Prom   =("Age",    "mean"),
).reset_index().rename(columns={"State_Name":"Estado"}))
state_summary["Fee_Promedio"] = state_summary["Fee_Promedio"].round(2)
state_summary["Pct_Gratis"]   = state_summary["Pct_Gratis"].round(1)
state_summary["Edad_Prom"]    = state_summary["Edad_Prom"].round(1)
state_summary["Lat"] = state_summary["Estado"].map(
    lambda s: STATE_COORDS.get(s,(np.nan,np.nan))[0])
state_summary["Lon"] = state_summary["Estado"].map(
    lambda s: STATE_COORDS.get(s,(np.nan,np.nan))[1])
state_summary["BubbleSize"] = (
    state_summary["Total"] / state_summary["Total"].max() * 55 + 8)

display(state_summary[["Estado","Total","Perros","Gatos",
                        "Fee_Promedio","Pct_Gratis","Edad_Prom"]])


,Estado,Total,Perros,Gatos,Fee_Promedio,Pct_Gratis,Edad_Prom
0,Johor,4,3,1,80.00,0.00,1.80
1,Kedah,73,47,26,22.67,83.60,13.00
2,Kelantan,63,39,24,27.90,73.00,25.30
3,Kuala Lumpur,1833,926,907,22.70,84.20,11.10
4,Labuan,510,282,228,23.50,89.00,10.70
5,Melaka,147,129,18,19.44,81.60,9.10
6,Negeri Sembilan,101,87,14,25.64,85.10,5.70
7,Pahang,17,6,11,14.71,82.40,5.50
8,Perak,127,63,64,19.53,88.20,10.00
9,Perlis,5,1,4,0.00,100.00,8.00


### 8.2 · Mapa de burbujas — Volumen de mascotas

In [34]:
fig = px.scatter_geo(
    state_summary, lat="Lat", lon="Lon",
    size="BubbleSize", size_max=55,
    color="Total", color_continuous_scale="Viridis",
    hover_name="Estado",
    hover_data={"Total":True,"Perros":True,"Gatos":True,
                "Fee_Promedio":True,"Pct_Gratis":True,
                "Lat":False,"Lon":False,"BubbleSize":False},
    text="Estado",
    title="🗺️ Mascotas en Adopción por Estado (Malasia)",
)
fig.update_traces(
    textposition="top center",
    marker=dict(opacity=0.85, line=dict(color="white", width=1.5)),
)
fig.update_geos(
    resolution=50,
    showcountries=True,  countrycolor="#AAAAAA",
    showcoastlines=True, coastlinecolor="#999999",
    showland=True,       landcolor="#EEF0E5",
    showocean=True,      oceancolor="#C8DFF5",
    lataxis_range=[-2,9], lonaxis_range=[98,120],
    projection_type="mercator",
)
fig.update_layout(height=560, paper_bgcolor="white",
                  margin=dict(l=0,r=0,t=50,b=10))
fig.show()


### 8.3 · Mapa de calor — Fee promedio

In [35]:
fig = px.scatter_geo(
    state_summary, lat="Lat", lon="Lon",
    size=[40]*len(state_summary), size_max=35,
    color="Fee_Promedio", color_continuous_scale="RdYlGn_r",
    hover_name="Estado",
    hover_data={"Fee_Promedio":True,"Pct_Gratis":True,
                "Total":True,"Lat":False,"Lon":False},
    text="Estado",
    title="🗺️ Fee Promedio de Adopción por Estado (Malasia)",
)
fig.update_traces(
    textposition="top center",
    marker=dict(opacity=0.85, line=dict(color="white", width=1.5)),
)
fig.update_geos(
    resolution=50,
    showcountries=True, showcoastlines=True,
    showland=True, landcolor="#EEF0E5",
    showocean=True, oceancolor="#C8DFF5",
    lataxis_range=[-2,9], lonaxis_range=[98,120],
    projection_type="mercator",
)
fig.update_layout(height=560, paper_bgcolor="white",
                  margin=dict(l=0,r=0,t=50,b=10))
fig.show()


### 8.4 · Totales y Distribución Perros/Gatos por Estado

In [36]:
fig = go.Figure(go.Bar(
    x=state_summary.sort_values("Total",ascending=False)["Estado"],
    y=state_summary.sort_values("Total",ascending=False)["Total"],
    marker=dict(color=state_summary.sort_values("Total",ascending=False)["Fee_Promedio"],
                colorscale="Teal", showscale=True,
                colorbar=dict(title="Fee Prom.")),
    text=state_summary.sort_values("Total",ascending=False)["Total"],
    textposition="outside",
))
fig.update_layout(title="Total de Mascotas en Adopción por Estado",
                  height=460, plot_bgcolor=BG_COLOR, paper_bgcolor="white",
                  xaxis_tickangle=-30)
fig.show()


In [37]:
fig = go.Figure()
fig.add_trace(go.Bar(x=state_summary["Estado"], y=state_summary["Perros"],
                     name="Perros 🐕", marker_color=COLOR_DOG))
fig.add_trace(go.Bar(x=state_summary["Estado"], y=state_summary["Gatos"],
                     name="Gatos 🐈", marker_color=COLOR_CAT))
fig.update_layout(barmode="stack",
                  title="Distribución Perros vs Gatos por Estado",
                  height=460, plot_bgcolor=BG_COLOR, paper_bgcolor="white",
                  xaxis_tickangle=-30)
fig.show()


## 🔗 9 · Matriz de Correlación

`Fee` muestra correlaciones bajas con la mayoría de variables. `PhotoAmt` es la más relevante positivamente.

In [38]:
# FIX: go.Heatmap directo — más compatible que px.imshow en todas las versiones de Plotly
numeric_cols = ["Age","Fee","Quantity","PhotoAmt","VideoAmt",
                "Vaccinated","Dewormed","Sterilized","Health",
                "MaturitySize","FurLength"]
corr = df[numeric_cols].corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.columns.tolist(),
    colorscale="RdBu_r",
    zmid=0, zmin=-1, zmax=1,
    text=corr.values,
    texttemplate="%{text}",
    textfont=dict(size=10),
    hoverongaps=False,
    colorbar=dict(title="Correlación"),
))
fig.update_layout(
    title="Matriz de Correlación entre Variables Numéricas",
    height=540, paper_bgcolor="white",
    xaxis=dict(tickangle=-30),
)
fig.show()


In [39]:
# Top correlaciones con Fee
corr_fee = (corr["Fee"].drop("Fee")
              .abs()
              .sort_values(ascending=False)
              .reset_index())
corr_fee.columns = ["Variable","Correlación |r| con Fee"]
display(corr_fee)


,Variable,Correlación |r| con Fee
0,FurLength,0.17
1,Vaccinated,0.14
2,Age,0.11
3,Dewormed,0.10
4,Sterilized,0.09
5,Quantity,0.05
6,PhotoAmt,0.03
7,Health,0.02
8,VideoAmt,0.01
9,MaturitySize,0.01


## 🔬 10 · Correlaciones Parciales

> **r simple:** relación entre 2 variables SIN controlar por las demás.  
> **r parcial (rp):** relación eliminando el efecto de todas las otras.  
> Si `r` alto y `rp` bajo → la relación era **espuria** (mediada por terceras variables).

**Método:** Matriz de precisión: `rp(i,j) = -P[i,j] / sqrt(P[i,i]*P[j,j])` donde P = corr⁻¹


In [40]:
import itertools

NUMERIC_COLS = ["Age","Fee","Quantity","PhotoAmt","VideoAmt",
                "Vaccinated","Dewormed","Sterilized","Health",
                "MaturitySize","FurLength"]

data_clean = df[NUMERIC_COLS].dropna()
corr_mat   = data_clean.corr()
prec       = np.linalg.inv(corr_mat.values)
n_var      = len(NUMERIC_COLS)

pcorr = np.zeros((n_var, n_var))
for i in range(n_var):
    for j in range(n_var):
        if i == j:
            pcorr[i, j] = 1.0
        else:
            denom = np.sqrt(prec[i,i] * prec[j,j])
            pcorr[i, j] = -prec[i,j] / denom if denom != 0 else 0.0

pcorr_df = pd.DataFrame(pcorr, index=NUMERIC_COLS, columns=NUMERIC_COLS).round(3)
print(f"Calculadas {n_var*(n_var-1)//2} pares de correlaciones parciales")


Calculadas 55 pares de correlaciones parciales


In [41]:
pairs = []
for i, j in itertools.combinations(range(n_var), 2):
    vi, vj = NUMERIC_COLS[i], NUMERIC_COLS[j]
    r_s  = round(corr_mat.loc[vi,vj], 3)
    r_p  = round(pcorr_df.loc[vi,vj], 3)
    diff = round(abs(r_s) - abs(r_p), 3)
    tipo = ("Espuria/Mediada" if diff > 0.10
            else "Directa"   if diff < -0.05
            else "Estable")
    pairs.append({"Par":f"{vi} <-> {vj}","r simple":r_s,"|r|":abs(r_s),
                  "r parcial":r_p,"|rp|":abs(r_p),"dif":diff,"Tipo":tipo})

pairs_df = pd.DataFrame(pairs).sort_values("|r|", ascending=False)
print("=== TOP 15 POR CORRELACION SIMPLE ===")
display(pairs_df.head(15)[["Par","r simple","r parcial","dif","Tipo"]].reset_index(drop=True))


=== TOP 15 POR CORRELACION SIMPLE ===


,Par,r simple,r parcial,dif,Tipo
0,Vaccinated <-> Dewormed,0.73,0.63,0.09,Estable
1,Vaccinated <-> Sterilized,0.51,0.27,0.24,Espuria/Mediada
2,Dewormed <-> Sterilized,0.47,0.16,0.31,Espuria/Mediada
3,Age <-> Sterilized,-0.18,-0.15,0.03,Estable
4,Fee <-> FurLength,0.17,0.17,0.00,Estable
5,Age <-> FurLength,0.15,0.15,0.00,Estable
6,Quantity <-> Dewormed,0.15,0.08,0.07,Estable
7,Quantity <-> Vaccinated,0.14,0.02,0.12,Espuria/Mediada
8,Quantity <-> PhotoAmt,0.14,0.14,0.01,Estable
9,Fee <-> Vaccinated,-0.14,-0.08,0.06,Estable


In [42]:
# Heatmap simple
fig = go.Figure(go.Heatmap(
    z=corr_mat.values, x=NUMERIC_COLS, y=NUMERIC_COLS,
    colorscale="RdBu_r", zmid=0, zmin=-1, zmax=1,
    text=corr_mat.round(2).values, texttemplate="%{text}",
    textfont=dict(size=9), colorbar=dict(title="r"), hoverongaps=False))
fig.update_layout(title="Correlacion Simple", height=520,
    paper_bgcolor="white", xaxis_tickangle=-35)
fig.show()


In [43]:
# Heatmap parcial
fig = go.Figure(go.Heatmap(
    z=pcorr_df.values, x=NUMERIC_COLS, y=NUMERIC_COLS,
    colorscale="RdBu_r", zmid=0, zmin=-1, zmax=1,
    text=pcorr_df.round(2).values, texttemplate="%{text}",
    textfont=dict(size=9), colorbar=dict(title="rp"), hoverongaps=False))
fig.update_layout(title="Correlacion Parcial (controlando todas las demas variables)",
    height=520, paper_bgcolor="white", xaxis_tickangle=-35)
fig.show()


In [44]:
# Top 10 pares: barras comparativas
top10 = pairs_df.head(10).copy().reset_index(drop=True)

fig = go.Figure()
fig.add_trace(go.Bar(name="r simple", x=top10["Par"], y=top10["r simple"],
    marker_color=[("#4E79A7" if v>=0 else "#E15759") for v in top10["r simple"]],
    text=[f"{v:+.3f}" for v in top10["r simple"]], textposition="outside"))
fig.add_trace(go.Bar(name="r parcial", x=top10["Par"], y=top10["r parcial"],
    marker_color=[("#59A14F" if v>=0 else "#F28E2B") for v in top10["r parcial"]],
    text=[f"{v:+.3f}" for v in top10["r parcial"]], textposition="outside"))
fig.add_hline(y=0, line_color="black", line_width=1)
fig.update_layout(barmode="group",
    title="Top 10 Pares: Correlacion Simple vs Parcial",
    xaxis_tickangle=-35, yaxis=dict(title="Correlacion", range=[-1.15,1.15]),
    height=520, plot_bgcolor=BG_COLOR, paper_bgcolor="white",
    legend=dict(x=0.75, y=0.98))
fig.show()


In [45]:
# Scatter: simple vs parcial (deteccion espurias)
fig = go.Figure()
color_tipo = {"Espuria/Mediada":"#E15759", "Estable":"#4E79A7", "Directa":"#59A14F"}
for tipo, grp in pairs_df.groupby("Tipo"):
    fig.add_trace(go.Scatter(
        x=grp["r simple"], y=grp["r parcial"],
        mode="markers+text", name=tipo,
        text=grp["Par"].str[:28],
        textposition="top center", textfont=dict(size=7),
        marker=dict(size=10, color=color_tipo[tipo],
                    line=dict(color="white",width=1)),
        hovertemplate="<b>%{text}</b><br>r=%{x:.3f}  rp=%{y:.3f}<extra></extra>"))
lim = 1.05
fig.add_trace(go.Scatter(x=[-lim,lim], y=[-lim,lim], mode="lines",
    name="r = rp (referencia)", line=dict(color="gray",dash="dash",width=1.5)))
fig.update_layout(
    title="Simple vs Parcial — Deteccion de Relaciones Espurias",
    xaxis=dict(title="r simple", range=[-lim,lim], zeroline=True),
    yaxis=dict(title="r parcial", range=[-lim,lim], zeroline=True),
    height=560, plot_bgcolor=BG_COLOR, paper_bgcolor="white",
    legend=dict(x=0.01, y=0.99))
fig.show()


In [46]:
# Tabla resumen por tipo
print("=== ESPURIAS/MEDIADAS ===")
display(pairs_df[pairs_df["Tipo"]=="Espuria/Mediada"][["Par","r simple","r parcial","dif"]].head(5).reset_index(drop=True))
print("\n=== ESTABLES ===")
display(pairs_df[pairs_df["Tipo"]=="Estable"][["Par","r simple","r parcial","dif"]].head(5).reset_index(drop=True))
print("\n=== DIRECTAS ===")
d_ = pairs_df[pairs_df["Tipo"]=="Directa"]
if len(d_) > 0:
    display(d_[["Par","r simple","r parcial","dif"]].head(5).reset_index(drop=True))
else:
    print("   No se encontraron pares con correlacion directa amplificada.")


=== ESPURIAS/MEDIADAS ===


,Par,r simple,r parcial,dif
0,Vaccinated <-> Sterilized,0.51,0.27,0.24
1,Dewormed <-> Sterilized,0.47,0.16,0.31
2,Quantity <-> Vaccinated,0.14,0.02,0.12



=== ESTABLES ===


,Par,r simple,r parcial,dif
0,Vaccinated <-> Dewormed,0.73,0.63,0.09
1,Age <-> Sterilized,-0.18,-0.15,0.03
2,Fee <-> FurLength,0.17,0.17,0.00
3,Age <-> FurLength,0.15,0.15,0.00
4,Quantity <-> Dewormed,0.15,0.08,0.07



=== DIRECTAS ===
   No se encontraron pares con correlacion directa amplificada.


## 💾 11 · Exportar Reporte HTML Completo

Genera un archivo **HTML interactivo y responsivo** con todos los gráficos embebidos.  
- No requiere internet para abrirse (Plotly via CDN, solo primera carga)  
- Todos los gráficos son interactivos (zoom, hover, filtros)  
- Salida: `C:\\Users\\juanc\\OneDrive\\Desktop\\EDA_Output\\EDA_Mascotas_Malaysia.html`

> Ejecutar **después** de haber corrido todas las celdas anteriores del notebook.


In [47]:
def fig_to_div(fig, div_id, first=False):
    return fig.to_html(
        full_html=False,
        include_plotlyjs="cdn" if first else False,
        div_id=div_id,
        config={"responsive":True,"displaylogo":False,
                "modeBarButtonsToRemove":["lasso2d","select2d"]})

print("Helper fig_to_div definido.")


Helper fig_to_div definido.


In [48]:
print("Regenerando figuras para HTML...")

# Dona tipo
tc = df["Type_Label"].value_counts().reset_index(); tc.columns=["T","C"]
F1 = go.Figure(go.Pie(labels=tc["T"],values=tc["C"],hole=0.45,
    marker=dict(colors=[COLOR_DOG,COLOR_CAT],line=dict(color="white",width=2)),
    textinfo="label+percent+value",textfont_size=13,pull=[0.03,0.03]))
F1.update_layout(title="Tipo de Mascota",height=420,paper_bgcolor="white",
    annotations=[dict(text="Pet",x=0.5,y=0.5,font_size=22,showarrow=False)])

# Genero bar
gc = df["Gender_Label"].value_counts().reset_index(); gc.columns=["G","C"]
F2 = go.Figure(go.Bar(x=gc["G"],y=gc["C"],marker_color=PALETTE[:3],
    text=gc["C"],textposition="outside"))
F2.update_layout(title="Genero",height=400,showlegend=False,
    plot_bgcolor=BG_COLOR,paper_bgcolor="white")

# Histograma edad
F3 = go.Figure()
F3.add_trace(go.Histogram(x=df[df["Type"]==1]["Age"],nbinsx=40,name="Perro",
    marker_color=COLOR_DOG,opacity=0.75))
F3.add_trace(go.Histogram(x=df[df["Type"]==2]["Age"],nbinsx=40,name="Gato",
    marker_color=COLOR_CAT,opacity=0.75))
F3.update_layout(barmode="overlay",title="Distribucion de Edad (meses)",
    height=420,plot_bgcolor=BG_COLOR,paper_bgcolor="white")

# Top 12 razas
br = df["Breed_Label"].value_counts().head(12).reset_index(); br.columns=["R","C"]
F4 = go.Figure(go.Bar(x=br["C"],y=br["R"],orientation="h",
    marker=dict(color=br["C"],colorscale="Blues",showscale=False),
    text=br["C"],textposition="outside"))
F4.update_layout(title="Top 12 Razas",height=460,plot_bgcolor=BG_COLOR,
    paper_bgcolor="white",yaxis=dict(autorange="reversed"))

# Colores
CH = {"Negro":"#2d2d2d","Marron":"#8B4513","Dorado":"#C8960C",
      "Amarillo":"#D4A017","Crema":"#A0785A","Gris":"#6B6B6B","Blanco":"#9E9E9E"}
cc = df["Color_Label"].value_counts().reset_index(); cc.columns=["C","N"]
F5 = go.Figure(go.Bar(x=cc["C"],y=cc["N"],
    marker=dict(color=[CH.get(c,"#999") for c in cc["C"]],
    line=dict(color="#222",width=1.5)),text=cc["N"],textposition="outside"))
F5.update_layout(title="Color Primario",height=420,showlegend=False,
    plot_bgcolor=BG_COLOR,paper_bgcolor="white")

# Histograma Fee
df2 = df[df["Fee"]<=600]
F6 = go.Figure()
F6.add_trace(go.Histogram(x=df2[df2["Type"]==1]["Fee"],nbinsx=50,name="Perro",
    marker_color=COLOR_DOG,opacity=0.75))
F6.add_trace(go.Histogram(x=df2[df2["Type"]==2]["Fee"],nbinsx=50,name="Gato",
    marker_color=COLOR_CAT,opacity=0.75))
F6.add_vline(x=df["Fee"].median(),line_dash="dash",line_color="red",
    annotation_text=f"Mediana: {df['Fee'].median():.0f} MYR")
F6.update_layout(barmode="overlay",title="Distribucion Fee (<= 600 MYR)",
    height=440,plot_bgcolor=BG_COLOR,paper_bgcolor="white")

# Fotos vs Fee tendencia manual
xv,yv = df["PhotoAmt"].values,df["Fee"].values
msk   = ~(np.isnan(xv)|np.isnan(yv))
coef  = np.polyfit(xv[msk],yv[msk],1)
xl    = np.linspace(xv[msk].min(),xv[msk].max(),100)
F7 = go.Figure()
for lbl,clr in [("Perro 🐕",COLOR_DOG),("Gato 🐈",COLOR_CAT)]:
    s = df[df["Type_Label"]==lbl]
    F7.add_trace(go.Scatter(x=s["PhotoAmt"],y=s["Fee"],mode="markers",name=lbl,
        marker=dict(color=clr,opacity=0.4,size=5)))
F7.add_trace(go.Scatter(x=xl,y=np.polyval(coef,xl),mode="lines",
    name=f"Tendencia (slope={coef[0]:.2f})",
    line=dict(color="black",dash="dash",width=2)))
F7.update_layout(title="Fotos vs Fee",height=440,
    plot_bgcolor=BG_COLOR,paper_bgcolor="white")

# Mapas
F8 = px.scatter_geo(state_summary,lat="Lat",lon="Lon",size="BubbleSize",size_max=55,
    color="Total",color_continuous_scale="Viridis",hover_name="Estado",text="Estado",
    hover_data={"Total":True,"Perros":True,"Gatos":True,"Fee_Promedio":True,
                "Pct_Gratis":True,"Lat":False,"Lon":False,"BubbleSize":False},
    title="Mapa - Mascotas por Estado")
F8.update_traces(textposition="top center",
    marker=dict(opacity=0.85,line=dict(color="white",width=1.5)))
F8.update_geos(resolution=50,showcountries=True,countrycolor="#AAA",
    showcoastlines=True,showland=True,landcolor="#EEF0E5",
    showocean=True,oceancolor="#C8DFF5",
    lataxis_range=[-2,9],lonaxis_range=[98,120],projection_type="mercator")
F8.update_layout(height=540,paper_bgcolor="white",margin=dict(l=0,r=0,t=50,b=10))

F9 = px.scatter_geo(state_summary,lat="Lat",lon="Lon",
    size=[40]*len(state_summary),size_max=35,
    color="Fee_Promedio",color_continuous_scale="RdYlGn_r",
    hover_name="Estado",text="Estado",title="Mapa - Fee Promedio")
F9.update_traces(textposition="top center",
    marker=dict(opacity=0.85,line=dict(color="white",width=1.5)))
F9.update_geos(resolution=50,showcountries=True,showcoastlines=True,
    showland=True,landcolor="#EEF0E5",showocean=True,oceancolor="#C8DFF5",
    lataxis_range=[-2,9],lonaxis_range=[98,120],projection_type="mercator")
F9.update_layout(height=540,paper_bgcolor="white",margin=dict(l=0,r=0,t=50,b=10))

# Correlacion simple
cm2 = df[NUMERIC_COLS].corr().round(2)
F10 = go.Figure(go.Heatmap(z=cm2.values,x=NUMERIC_COLS,y=NUMERIC_COLS,
    colorscale="RdBu_r",zmid=0,zmin=-1,zmax=1,
    text=cm2.values,texttemplate="%{text}",textfont=dict(size=9),
    colorbar=dict(title="r")))
F10.update_layout(title="Correlacion Simple",height=520,
    paper_bgcolor="white",xaxis_tickangle=-35)

# Correlacion parcial
F11 = go.Figure(go.Heatmap(z=pcorr_df.values,x=NUMERIC_COLS,y=NUMERIC_COLS,
    colorscale="RdBu_r",zmid=0,zmin=-1,zmax=1,
    text=pcorr_df.round(2).values,texttemplate="%{text}",textfont=dict(size=9),
    colorbar=dict(title="rp")))
F11.update_layout(title="Correlacion Parcial",height=520,
    paper_bgcolor="white",xaxis_tickangle=-35)

# Simple vs parcial barras
t10 = pairs_df.head(10).copy().reset_index(drop=True)
F12 = go.Figure()
F12.add_trace(go.Bar(name="r simple",x=t10["Par"],y=t10["r simple"],
    marker_color=[("#4E79A7" if v>=0 else "#E15759") for v in t10["r simple"]],
    text=[f"{v:+.3f}" for v in t10["r simple"]],textposition="outside"))
F12.add_trace(go.Bar(name="r parcial",x=t10["Par"],y=t10["r parcial"],
    marker_color=[("#59A14F" if v>=0 else "#F28E2B") for v in t10["r parcial"]],
    text=[f"{v:+.3f}" for v in t10["r parcial"]],textposition="outside"))
F12.add_hline(y=0,line_color="black",line_width=1)
F12.update_layout(barmode="group",title="Simple vs Parcial Top 10",
    xaxis_tickangle=-35,yaxis=dict(range=[-1.15,1.15]),
    height=520,plot_bgcolor=BG_COLOR,paper_bgcolor="white")

print("Todas las figuras listas.")


Regenerando figuras para HTML...
Todas las figuras listas.


In [49]:
# KPIs
tot_h  = len(df)
dog_h  = (df["Type"]==1).sum()
cat_h  = (df["Type"]==2).sum()
free_h = round((df["Fee"]==0).mean()*100,1)
afee_h = round(df.loc[df["Fee"]>0,"Fee"].mean(),2)
sta_h  = df["State_Name"].nunique()
hlt_h  = round((df["Health"]==1).mean()*100,1)

CC = ["#4E79A7","#F28E2B","#59A14F","#E15759",
      "#76B7B2","#EDC948","#B07AA1","#FF9DA7"]

def mk(icon,title,value,color,sub=""):
    s = f"<div class='card-sub'>{sub}</div>" if sub else ""
    return (f"<div class='card' style='border-top:4px solid {color}'>"
            f"<div class='card-icon'>{icon}</div>"
            f"<div class='card-value' style='color:{color}'>{value}</div>"
            f"<div class='card-title'>{title}</div>{s}</div>")

KPIS_HTML = (
    mk("🐾","Total Mascotas",f"{tot_h:,}",CC[0]) +
    mk("🐕","Perros",f"{dog_h:,}",CC[1],f"{dog_h/tot_h*100:.1f}%") +
    mk("🐈","Gatos",f"{cat_h:,}",CC[2],f"{cat_h/tot_h*100:.1f}%") +
    mk("🆓","Adopciones Gratis",f"{free_h}%",CC[3]) +
    mk("💰","Fee Prom c/cargo",f"{afee_h} MYR",CC[4]) +
    mk("💚","Mascotas Sanas",f"{hlt_h}%",CC[5]) +
    mk("📍","Estados",f"{sta_h}",CC[6],"Malasia") +
    mk("🔬","Pares Analizados",f"{len(pairs_df)}",CC[7],"Corr. parciales")
)
print("KPIs listos.")


KPIs listos.


In [50]:
CSS_STR = (
    "*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}\n"
    ":root{--bg:#F0F4F8;--surf:#fff;--primary:#2C3E6E;--accent:#4E79A7;"
    "--text:#2D3748;--muted:#718096;--r:12px;--sh:0 2px 12px rgba(0,0,0,.08)}\n"
    "html{scroll-behavior:smooth}\n"
    "body{font-family:'Segoe UI',system-ui,sans-serif;background:var(--bg);"
    "color:var(--text);line-height:1.6}\n"
    "header{background:linear-gradient(135deg,#1a2a4a,#2C3E6E 50%,#4E79A7);"
    "color:#fff;padding:3rem 2rem 2.5rem;text-align:center}\n"
    "header h1{font-size:clamp(1.6rem,4vw,2.8rem);font-weight:700}\n"
    ".badge{display:inline-block;margin-top:1rem;background:rgba(255,255,255,.15);"
    "border:1px solid rgba(255,255,255,.3);padding:.3rem .9rem;"
    "border-radius:20px;font-size:.85rem}\n"
    "nav{position:sticky;top:0;z-index:100;background:var(--primary);"
    "padding:.6rem 1rem;display:flex;flex-wrap:wrap;gap:.4rem;"
    "justify-content:center;box-shadow:0 2px 8px rgba(0,0,0,.2)}\n"
    "nav a{color:rgba(255,255,255,.85);text-decoration:none;padding:.35rem .9rem;"
    "border-radius:20px;font-size:.85rem;transition:background .2s}\n"
    "nav a:hover{background:rgba(255,255,255,.2);color:#fff}\n"
    "main{max-width:1300px;margin:2rem auto;padding:0 1rem 4rem}\n"
    ".kpi-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));"
    "gap:1rem;margin-bottom:2.5rem}\n"
    ".card{background:var(--surf);border-radius:var(--r);padding:1.2rem 1rem;"
    "box-shadow:var(--sh);text-align:center;transition:transform .2s}\n"
    ".card:hover{transform:translateY(-3px)}\n"
    ".card-icon{font-size:1.8rem}\n"
    ".card-value{font-size:1.7rem;font-weight:700;margin:.3rem 0}\n"
    ".card-title{font-size:.78rem;color:var(--muted);text-transform:uppercase;"
    "letter-spacing:.05em}\n"
    ".card-sub{font-size:.75rem;color:var(--muted);margin-top:.2rem}\n"
    ".eda-section{background:var(--surf);border-radius:var(--r);"
    "padding:1.8rem 1.5rem;margin-bottom:2rem;box-shadow:var(--sh)}\n"
    ".section-header{display:flex;align-items:flex-start;gap:1rem;"
    "margin-bottom:1.5rem;padding-bottom:1rem;border-bottom:2px solid var(--bg)}\n"
    ".section-icon{font-size:2rem;line-height:1}\n"
    ".eda-section h2{font-size:1.25rem;color:var(--primary);font-weight:700}\n"
    ".section-desc{color:var(--muted);font-size:.9rem;margin-top:.2rem}\n"
    ".charts-grid{display:grid;"
    "grid-template-columns:repeat(auto-fit,minmax(min(100%,480px),1fr));gap:1.2rem}\n"
    ".chart-box{border:1px solid #e8ecf0;border-radius:8px;"
    "overflow:hidden;background:#fafbfc}\n"
    ".chart-box.full{grid-column:1/-1}\n"
    ".insights{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));"
    "gap:.8rem;margin-top:1.2rem}\n"
    ".insight{border-left:4px solid var(--accent);background:#f0f5ff;"
    "padding:.8rem 1rem;border-radius:0 8px 8px 0;font-size:.875rem}\n"
    ".insight strong{color:var(--primary)}\n"
    "footer{text-align:center;padding:2rem;color:var(--muted);font-size:.85rem;"
    "border-top:1px solid #dde3ea;margin-top:2rem}\n"
    "@media(max-width:600px){"
    ".kpi-grid{grid-template-columns:repeat(2,1fr)}"
    ".chart-box.full{grid-column:span 1}}\n"
)
print("CSS listo.")


CSS listo.


In [51]:
# Generar divs (first=True solo en el primero)
d1  = fig_to_div(F1,  "d01", first=True)
d2  = fig_to_div(F2,  "d02")
d3  = fig_to_div(F3,  "d03")
d4  = fig_to_div(F4,  "d04")
d5  = fig_to_div(F5,  "d05")
d6  = fig_to_div(F6,  "d06")
d7  = fig_to_div(F7,  "d07")
d8  = fig_to_div(F8,  "d08")
d9  = fig_to_div(F9,  "d09")
d10 = fig_to_div(F10, "d10")
d11 = fig_to_div(F11, "d11")
d12 = fig_to_div(F12, "d12")

# Ensamblar HTML
HTML_PARTS = [
    "<!DOCTYPE html>\n<html lang='es'>\n<head>\n",
    "<meta charset='UTF-8'/>\n",
    "<meta name='viewport' content='width=device-width,initial-scale=1.0'/>\n",
    "<title>EDA Mascotas Malasia</title>\n",
    f"<style>{CSS_STR}</style>\n</head>\n<body>\n",
    # Header
    "<header>\n",
    "<h1>Adopcion de Mascotas en Malasia</h1>\n",
    "<p>Analisis Exploratorio Completo - Dataset PetFinder Malaysia</p>\n",
    f"<div class='badge'>{tot_h:,} registros | {sta_h} estados | Fee (MYR) | Correlaciones parciales</div>\n",
    "</header>\n",
    # Nav
    "<nav>\n",
    "<a href='#res'>Resumen</a>\n",
    "<a href='#dis'>Distribucion</a>\n",
    "<a href='#fee'>Fee</a>\n",
    "<a href='#map'>Mapas</a>\n",
    "<a href='#cor'>Correlacion</a>\n",
    "<a href='#par'>Parcial</a>\n",
    "</nav>\n",
    # Main
    "<main>\n",
    f"<div id='res' style='padding-top:1rem'><div class='kpi-grid'>{KPIS_HTML}</div></div>\n",
    # Distribucion
    "<section id='dis' class='eda-section'>\n",
    "<div class='section-header'><span class='section-icon'>Distribucion</span>",
    "<div><h2>Distribucion General</h2>",
    "<p class='section-desc'>Tipo, genero, edad, razas y colores.</p></div></div>\n",
    "<div class='charts-grid'>\n",
    f"<div class='chart-box'>{d1}</div>\n",
    f"<div class='chart-box'>{d2}</div>\n",
    f"<div class='chart-box full'>{d3}</div>\n",
    f"<div class='chart-box full'>{d4}</div>\n",
    f"<div class='chart-box full'>{d5}</div>\n",
    "</div></section>\n",
    # Fee
    "<section id='fee' class='eda-section'>\n",
    "<div class='section-header'><span class='section-icon'>Fee</span>",
    "<div><h2>Variable Objetivo: Fee</h2>",
    "<p class='section-desc'>Distribucion y relacion con fotos.</p></div></div>\n",
    "<div class='charts-grid'>\n",
    f"<div class='chart-box full'>{d6}</div>\n",
    f"<div class='chart-box full'>{d7}</div>\n",
    "</div></section>\n",
    # Mapas
    "<section id='map' class='eda-section'>\n",
    "<div class='section-header'><span class='section-icon'>Mapas</span>",
    f"<div><h2>Analisis por Estado</h2>",
    f"<p class='section-desc'>{sta_h} estados de Malasia.</p></div></div>\n",
    "<div class='charts-grid'>\n",
    f"<div class='chart-box full'>{d8}</div>\n",
    f"<div class='chart-box full'>{d9}</div>\n",
    "</div></section>\n",
    # Correlacion simple
    "<section id='cor' class='eda-section'>\n",
    "<div class='section-header'><span class='section-icon'>Correlacion</span>",
    "<div><h2>Correlacion Simple</h2>",
    "<p class='section-desc'>Pearson entre variables numericas.</p></div></div>\n",
    "<div class='charts-grid'>\n",
    f"<div class='chart-box full'>{d10}</div>\n",
    "</div></section>\n",
    # Correlacion parcial
    "<section id='par' class='eda-section'>\n",
    "<div class='section-header'><span class='section-icon'>Parcial</span>",
    "<div><h2>Correlaciones Parciales</h2>",
    "<p class='section-desc'>Simple vs Parcial - deteccion de relaciones espurias.</p></div></div>\n",
    "<div class='charts-grid'>\n",
    f"<div class='chart-box full'>{d11}</div>\n",
    f"<div class='chart-box full'>{d12}</div>\n",
    "</div>\n",
    "<div class='insights'>\n",
    "<div class='insight'><strong>Espuria/Mediada:</strong> r simple alta, rp baja. Terceras variables explican la relacion.</div>\n",
    "<div class='insight'><strong>Estable:</strong> r aprox rp. Relacion directa confirmada.</div>\n",
    "<div class='insight'><strong>Directa:</strong> rp mayor que r. Relacion subestimada sin control.</div>\n",
    "</div></section>\n",
    "</main>\n",
    f"<footer><p>EDA con Python + Plotly | PetFinder Malaysia | {tot_h:,} registros</p></footer>\n",
    "</body>\n</html>"
]

HTML_FILE_PATH = r"C:\\Users\\juanc\\OneDrive\\Desktop\\EDA_Output\\EDA_Mascotas_Malaysia.html"
with open(HTML_FILE_PATH, "w", encoding="utf-8") as fh:
    fh.write("".join(HTML_PARTS))

size_kb = os.path.getsize(HTML_FILE_PATH) / 1024
print(f"HTML generado correctamente")
print(f"Ruta  : {HTML_FILE_PATH}")
print(f"Tamano: {size_kb:.0f} KB")
print(f"Graficos embebidos: 12 (incluyendo correlaciones parciales)")
print(f"Abrir en Chrome o Firefox para visualizar.")


HTML generado correctamente
Ruta  : C:\\Users\\juanc\\OneDrive\\Desktop\\EDA_Output\\EDA_Mascotas_Malaysia.html
Tamano: 185 KB
Graficos embebidos: 12 (incluyendo correlaciones parciales)
Abrir en Chrome o Firefox para visualizar.


## 📌 12 · Conclusiones Principales

| # | Hallazgo | Detalle |
|---|---|---|
| 1 | **85.5% adopciones gratuitas** | Mediana Fee = 0 MYR |
| 2 | **Fee promedio con cargo** | ~150 MYR (~33 USD) |
| 3 | **Kuala Lumpur concentra el 46%** | 1.833 de 3.972 mascotas |
| 4 | **Perros ligeramente mas comunes** | 52.9% vs 47.1% gatos |
| 5 | **Altisimo nivel de salud** | 96.2% de mascotas saludables |
| 6 | **Mascotas jovenes dominan** | Mediana 4 meses de edad |
| 7 | **Mas fotos → fee levemente mas alto** | Correlacion positiva leve |
| 8 | **Negro es el color mas frecuente** | 50.1% del total |
| 9 | **Mixed Breed domina las razas** | 37.7% de los listings |
| 10 | **Vacunacion/Despar./Esteril. son espurias entre si** | r parciales mucho mas bajas que las simples |
| 11 | **Fee tiene r parciales muy bajas con todo** | Decision autonoma del rescatista, no predecible por atributos de la mascota |
